# nanoPeriGPT — Incremental Experiments

Peridynamic attention + DEER layer-parallelism for transformers.

Each cell runs one experiment. Compare val_loss between cells to track progress.

In [ ]:
# Setup: upload project files or clone from repo
import os
import torch

# If running from uploaded zip:
# !unzip -q perigpt.zip -d perigpt
# os.chdir('perigpt')

# If cloning from git:
# !git clone <your-repo-url> perigpt
# os.chdir('perigpt')

print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name()}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
    print(f'BF16: {torch.cuda.is_bf16_supported()}')

In [ ]:
# Prepare data
!python data/shakespeare_char/prepare.py

In [ ]:
# Phase 1: Vanilla nanoGPT Baseline
!python train.py config/baseline.py

In [ ]:
# Phase 2: Bond-Based Peridynamic Attention (horizon=32)
!python train.py config/baseline.py \
    --out_dir=out-peri-h32 \
    --wandb_run_name=peri_h32 \
    --experiment_name=peri_h32 \
    --attention_type=peridynamic \
    --horizon=32

In [ ]:
# Phase 3: Horizon Sweep
for h in [16, 64, 128]:
    print(f'\n{"="*60}')
    print(f'Horizon = {h}')
    print(f'{"="*60}')
    !python train.py config/baseline.py \
        --out_dir=out-peri-h{h} \
        --wandb_run_name=peri_h{h} \
        --experiment_name=peri_h{h} \
        --attention_type=peridynamic \
        --horizon={h}

In [ ]:
# Check results so far
import pandas as pd
df = pd.read_csv('results.tsv', sep='\t')
print(df.sort_values('val_loss').to_string(index=False))

In [ ]:
# Phase 5: Block Attention Residual (standard attn + block_attn depth)
!python train.py config/baseline.py \
    --out_dir=out-blockres \
    --wandb_run_name=block_attn_res \
    --experiment_name=block_attn_res \
    --residual_type=block_attn \
    --depth_block_size=3

In [ ]:
# Phase 6: Combined — Peri Attention + Block Attn Residual
# Use best horizon from Phase 3
BEST_HORIZON = 32  # <-- update based on Phase 3 results
!python train.py config/baseline.py \
    --out_dir=out-peri-blockres \
    --wandb_run_name=peri_blockres \
    --experiment_name=peri_blockres \
    --attention_type=peridynamic \
    --horizon={BEST_HORIZON} \
    --residual_type=block_attn \
    --depth_block_size=3

In [ ]:
# Phase 11: Hybrid Attention (Peri local + sparse global)
!python train.py config/baseline.py \
    --out_dir=out-hybrid \
    --wandb_run_name=hybrid_a8 \
    --experiment_name=hybrid_a8 \
    --attention_type=hybrid \
    --horizon={BEST_HORIZON} \
    --n_global_anchors=8

In [ ]:
# Final results comparison
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('results.tsv', sep='\t')
df = df.sort_values('val_loss')

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['green' if s == 'KEEP' else 'red' for s in df['status']]
ax.barh(df['experiment'], df['val_loss'], color=colors)
ax.set_xlabel('Validation Loss')
ax.set_title('nanoPeriGPT Experiments — shakespeare_char')
ax.invert_yaxis()
for i, (v, t) in enumerate(zip(df['val_loss'], df['train_loss'])):
    ax.text(v + 0.001, i, f'{v:.4f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('progress.png', dpi=150)
plt.show()
print(df.to_string(index=False))

In [ ]:
# Sample from best model
BEST_DIR = 'out-baseline'  # <-- update to best experiment's out_dir
!python sample.py --out_dir={BEST_DIR} --num_samples=3 --max_new_tokens=500